# Day 2: Inventory data (solution)

Reference solutions. Every cell is runnable; `practice.ipynb` is the blank-page twin.

## `s1_data.py`: Data — nested dicts, merge, sort records.

In [ ]:
inv = {"apple": {"qty": 5, "price": 0.5}, "pear": {"qty": 0, "price": 0.75}}
inv["kiwi"] = {"qty": 12, "price": 0.2}
merged = inv | {"apple": {"qty": 9, "price": 0.5}}

In [ ]:
assert merged["apple"]["qty"] == 9 and inv["apple"]["qty"] == 5
assert round(sum(v["qty"] * v["price"] for v in inv.values()), 2) == 4.9

In [ ]:
rows = sorted(((k, v["qty"]) for k, v in inv.items()), key=lambda r: -r[1])

In [ ]:
assert rows[0] == ("kiwi", 12)

**Then, vary it:**

- VARIATION: use dict.pop with a default to remove "pear" safely
- VARIATION: invert to {qty: [names]} with setdefault

## `s2_flow.py`: Flow — filter dicts, any/all, for-else, walrus, zip/enumerate.

In [ ]:
inv = {"apple": 5, "pear": 0, "kiwi": 12}
in_stock = {k: v for k, v in inv.items() if v}

In [ ]:
assert set(in_stock) == {"apple", "kiwi"}
assert any(v == 0 for v in inv.values()) and not all(inv.values())

In [ ]:
for name, q in inv.items():
    if q > 10:
        found = name
        break
else:
    found = None

In [ ]:
assert found == "kiwi"
if (n := len(in_stock)) > 1: assert n == 2

In [ ]:
names = list(inv)

In [ ]:
assert list(zip(names, range(3)))[1] == ("pear", 1)
assert next(enumerate(names, 1)) == (1, "apple")

**Then, vary it:**

- VARIATION: change the threshold so for-else takes the else branch
- VARIATION: use zip(strict=True) with mismatched lengths and catch the error

## `s3_functions.py`: Functions — defaults, *args/**kw, partial, sort key.

In [ ]:
from functools import partial
def restock(inv, name, qty=1, *, log=None):
    log = [] if log is None else log   # never use log=[] as a default
    inv[name] = inv.get(name, 0) + qty
    log.append(name)
    return log
inv = {}
restock(inv, "a"); restock(inv, "a", qty=4)


In [ ]:
assert inv == {"a": 5}

In [ ]:
def total(*qtys, **named): return sum(qtys) + sum(named.values())

In [ ]:
assert total(1, 2, x=3) == 6

In [ ]:
partial(restock, qty=10)(inv, "b")

In [ ]:
assert sorted(inv, key=inv.get, reverse=True) == ["b", "a"]

**Then, vary it:**

- VARIATION: change log=None to log=[] and call twice to see the shared-default bug
- VARIATION: make restock positional-only for inv using `/`

## `s4_classes.py`: Classes — dataclass fields, __post_init__, property, classmethod.

In [ ]:
from dataclasses import dataclass, field
@dataclass
class Item:
    name: str
    qty: int = 0
    price: float = 0.0
    tags: list[str] = field(default_factory=list)
    def __post_init__(self):
        if self.qty < 0: raise ValueError("qty < 0")
    @property
    def value(self): return self.qty * self.price
    @classmethod
    def from_csv(cls, line):
        n, q, p = line.split(",")
        return cls(n, int(q), float(p))

In [ ]:
assert Item.from_csv("apple,5,0.5").value == 2.5
assert Item("x").tags is not Item("y").tags

In [ ]:
try: Item("bad", -1)
except ValueError: pass
else: raise AssertionError

**Then, vary it:**

- VARIATION: add a staticmethod `is_valid_name`
- VARIATION: make Item frozen and replace() it with dataclasses.replace

## `s5_errors.py`: Errors — raise from, JSON persistence.

In [ ]:
import json, tempfile
from pathlib import Path
class OutOfStock(Exception): pass
def take(inv, name, q):
    try: have = inv[name]
    except KeyError as e: raise OutOfStock(name) from e
    if have < q: raise OutOfStock(name)
    inv[name] = have - q
inv = {"a": 2}
take(inv, "a", 1)

In [ ]:
assert inv["a"] == 1

In [ ]:
for bad in ("a", "zzz"):
    try: take(inv, bad, 5)
    except OutOfStock: pass
    else: raise AssertionError

In [ ]:
from pathlib import Path
import json, tempfile

with tempfile.TemporaryDirectory() as d:
    p = Path(d) / "inv.json"
    p.write_text(json.dumps(inv))
    assert json.loads(p.read_text()) == inv

**Then, vary it:**

- VARIATION: catch OutOfStock and inspect e.__cause__
- VARIATION: add try/else/finally and log which branch ran

## `s6_stdlib.py`: Stdlib — namedtuple, deque, groupby, lru_cache, date math.

In [ ]:
from collections import namedtuple, deque
from functools import lru_cache
from itertools import groupby
from datetime import date, timedelta
P = namedtuple("P", "name qty")

In [ ]:
assert P("a", 1)._replace(qty=2).qty == 2

In [ ]:
d = deque(maxlen=2)
for i in range(4): d.append(i)

In [ ]:
assert list(d) == [2, 3]

In [ ]:
rows = sorted([("fruit", "apple"), ("veg", "kale"), ("fruit", "pear")])
g = {k: [n for _, n in v] for k, v in groupby(rows, key=lambda r: r[0])}

In [ ]:
assert g == {"fruit": ["apple", "pear"], "veg": ["kale"]}

In [ ]:
@lru_cache
def fib(n): return n if n < 2 else fib(n - 1) + fib(n - 2)

In [ ]:
assert fib(50) == 12586269025

In [ ]:
assert date(2026, 1, 30) + timedelta(days=3) == date(2026, 2, 2)

**Then, vary it:**

- VARIATION: call fib.cache_info() and print it
- VARIATION: groupby without sorting first; see why the result breaks

## `s7_concurrency.py`: Concurrency — shared state needs a Lock.

In [ ]:
import threading
inv = {"a": 0}
lock = threading.Lock()
def add():
    for _ in range(1000):
        with lock: inv["a"] += 1
ts = [threading.Thread(target=add) for _ in range(4)]
for t in ts: t.start()
for t in ts: t.join()

In [ ]:
assert inv["a"] == 4000

**Then, vary it:**

- VARIATION: remove the lock and increase iterations; does it ever lose updates?
- VARIATION: replace the lock with queue.Queue and a single consumer

## `s8_quality.py`: Quality — TypedDict, table-driven checks.

In [ ]:
from typing import TypedDict
class Row(TypedDict):
    name: str
    qty: int
def low(rows: list[Row], limit: int = 3) -> list[str]:
    return [r["name"] for r in rows if r["qty"] < limit]
cases = [([], []), ([{"name": "a", "qty": 1}], ["a"]), ([{"name": "a", "qty": 9}], [])]

In [ ]:
for rows, want in cases:
    assert low(rows) == want, (rows, want)

**Then, vary it:**

- VARIATION: add a boundary case (qty == limit)
- VARIATION: add NotRequired[str] field "note" to Row

## `kata.py`: KATA — low-stock report.

In [ ]:
from dataclasses import dataclass
@dataclass
class Item:
    name: str
    qty: int
    price: float
def report(items, limit=5):
    low = sorted((i for i in items if i.qty < limit), key=lambda i: (i.qty, i.name))
    return [f"{i.name}: {i.qty}" for i in low]
items = [Item("pear", 0, .7), Item("apple", 3, .5), Item("kiwi", 12, .2), Item("fig", 3, 1)]

In [ ]:
assert report(items) == ["pear: 0", "apple: 3", "fig: 3"]

**Then, vary it:**

- KATA: rewrite from memory; add total stock value line at the end